<a href="https://colab.research.google.com/github/VincentCCL/MTAT/blob/main/notebooks/MTAT2026_BLEURT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3.4.4 BLEURT

We have started this new session with Runtime = GPU. While this is not strictly necessary, using CPU (the default) it will take much longer to run a BLEURT evaluation.

First, we install:

In [ ]:
!pip install -q git+https://github.com/google-research/bleurt.git
!pip install -q huggingface_hub

  Preparing metadata (setup.py) ... done


We download the model.

In [ ]:
from huggingface_hub import snapshot_download

ckpt_dir = snapshot_download(
    repo_id="BramVanroy/BLEURT-20",
    local_dir="bleurt-20",
    local_dir_use_symlinks=False
)
print("Checkpoint in:", ckpt_dir)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Checkpoint in: /content/bleurt-20


Then we define a function that makes Bleurt calculation easy and tells us what the system is doing, as this may be quite time sensitive.

In [ ]:
import time
from bleurt import score
from statistics import mean

def calculate_bleurt(ref_file, mt_file):
    t0 = time.time()
    print("Reading files...", flush=True)

    with open(ref_file, encoding="utf-8") as f:
        refs = [line.strip() for line in f]
    with open(mt_file, encoding="utf-8") as f:
        cands = [line.strip() for line in f]

    if len(refs) != len(cands):
        raise ValueError(f"Line mismatch: {len(refs)} refs vs {len(cands)} candidates")

    print(f"Loaded {len(refs)} pairs in {time.time()-t0:.1f}s", flush=True)

    t1 = time.time()
    print("Loading BLEURT model (this can take a while the first time)...", flush=True)
    scorer = score.BleurtScorer("bleurt-20")
    print(f"Model loaded in {time.time()-t1:.1f}s", flush=True)

    t2 = time.time()
    print("Scoring...", flush=True)
    scores = scorer.score(references=refs, candidates=cands)
    print(f"Scored in {time.time()-t2:.1f}s", flush=True)

    print("BLEURT mean:", mean(scores))

To test it, we can download the `tatoeba-bing.txt` file which contains the bing translation of the first 30 sentences in `dev.nl`.  We also download the latter and make a reduced version using the `head -n 30` command, that results in the first 30 lines of the `dev.nl`. The result is written to `dev30.nl`.

In [ ]:
!wget https://raw.githubusercontent.com/VincentCCL/MTAT/refs/heads/main/data/tatoeba-en-nl/tatoeba-bing.txt
!wget https://raw.githubusercontent.com/VincentCCL/MTAT/refs/heads/main/data/tatoeba-en-nl/dev.nl

--2026-03-05 11:37:12--  https://raw.githubusercontent.com/VincentCCL/MTAT/refs/heads/main/data/tatoeba-en-nl/tatoeba-bing.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 989 [text/plain]
Saving to: ‘tatoeba-bing.txt’

tatoeba-bing.txt    100%[===================>]     989  --.-KB/s    in 0s      

2026-03-05 11:37:13 (59.2 MB/s) - ‘tatoeba-bing.txt’ saved [989/989]

--2026-03-05 11:37:13--  https://raw.githubusercontent.com/VincentCCL/MTAT/refs/heads/main/data/tatoeba-en-nl/dev.nl
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Lengt

In [ ]:
!head -n 30 dev.nl > dev30.nl

And then for the actual calculation:

In [ ]:
calculate_bleurt('dev30.nl','tatoeba-bing.txt')

Reading files...
Loaded 30 pairs in 0.0s
Loading BLEURT model (this can take a while the first time)...
Model loaded in 10.7s
Scoring...
Scored in 8.6s
BLEURT mean: 0.8318437298138937
